# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [13]:
import pandas as pd
import plotly.express as px
import numpy as np

# ── Load dataset ─────────────────────────────────────────────
energy_df = pd.read_csv('global_energy_mix.csv')

# ── Define energy source categories ──────────────────────────
energy_types = {
    'Coal': 'Fossil',
    'Oil': 'Fossil',
    'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon',
    'Hydro': 'Low-carbon',
    'Wind': 'Renewable',
    'Solar': 'Renewable',
    'Other Renewables': 'Renewable'
}

# Assign categories to each energy source
energy_df['Source_Type'] = energy_df['Source'].replace(energy_types)

# Check the dataset
print(f"Total records: {energy_df.shape[0]}")
print(energy_df.head(10))

Total records: 103
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [14]:
# ── Filter fossil fuel data ──────────────────────────────────
fossil_df = energy_df[energy_df['Source_Type'] == 'Fossil'].copy()

# Calculate total usage by region and country
region_sum = fossil_df.groupby('Region')['TWh'].sum()
country_sum = fossil_df.groupby('Country')['TWh'].sum()

highest_region = region_sum.idxmax()
highest_country = country_sum.idxmax()

# ── Step 1: Create Treemap ───────────────────────────────────
fig = px.treemap(
    fossil_df,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map={
        'Coal': '#0072B2',
        'Oil': '#E69F00',
        'Natural Gas': '#009E73'
    },
    title=f'Fossil Fuel Consumption Across {highest_region} and {highest_country}'
)

# ── Step 2: Customize Treemap ────────────────────────────────
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    marker=dict(
        colors=[
            {'Coal': '#0072B2',
             'Oil': '#E69F00',
             'Natural Gas': '#009E73'}.get(label, '#BDBDBD')
            for label in fig.data[0].labels
        ]
    )
)

fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=5, r=5, t=55, b=5)
)

# Display Treemap
fig.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [15]:
# ── Filter low-carbon energy ─────────────────────────────────
clean_energy = energy_df[energy_df['Source_Type'] == 'Low-carbon'].copy()

# Calculate total energy production by country
country_totals = clean_energy.groupby(
    'Country', as_index=False
)['TWh'].sum()

country_totals = country_totals.sort_values(
    'TWh', ascending=True
)

leading_country = country_totals.iloc[-1]['Country']

country_totals['Category'] = 'Low-carbon'

# ── Step 1: Create Treemap ────────────────────────────────────
fig = px.treemap(
    country_totals,
    path=['Category', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Viridis',
    title='Country-wise Distribution of Low-Carbon Energy'
)

fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh'
)

fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=5, r=5, t=55, b=5)
)

fig.show()


# ── Step 2: Create Horizontal Bar Chart ──────────────────────
fig = px.bar(
    country_totals,
    x='TWh',
    y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=['#009E73'],
    title=f'{leading_country} Has the Highest Low-Carbon Energy Production'
)

fig.update_traces(
    texttemplate='%{x:,.0f} TWh',
    textposition='outside'
)

fig.update_layout(
    font=dict(family='Arial', size=12),
    xaxis_title='Total Energy (TWh)',
    yaxis_title='Country',
    margin=dict(l=10, r=40, t=55, b=10)
)

fig.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The bar chart is more effective for showing the ranking because it arranges countries according to their low-carbon energy production. It allows us to quickly see which countries produce more energy and compare their values. The treemap provides a good overview of how energy production is distributed, but it is harder to identify the exact ranking of countries.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

The treemap helps identify which individual countries consume the largest amounts of fossil fuels and which energy sources contribute most to their consumption. The stacked bar chart makes it easier to compare the fuel composition across regions and determine where coal represents the largest percentage of fossil fuel usage. Therefore, the treemap is useful for comparing consumption volumes, while the stacked bar chart is better for comparing regional energy dependency.

In [16]:
# ── Filter fossil fuel data ───────────────────────────────────
fossil_df = energy_df[energy_df['Source_Type'] == 'Fossil'].copy()

# Calculate energy usage by region and source
region_summary = fossil_df.groupby(
    ['Region', 'Source'], as_index=False
)['TWh'].sum()

# Calculate percentage share for each region
region_summary['Region_Total'] = region_summary.groupby(
    'Region'
)['TWh'].transform('sum')

region_summary['Percentage'] = (
    region_summary['TWh'] / region_summary['Region_Total']
) * 100

# Find the region with the highest coal share
coal_summary = region_summary[region_summary['Source'] == 'Coal'].sort_values(
    'Percentage', ascending=False
)

highest_region = coal_summary.iloc[0]['Region']
highest_share = coal_summary.iloc[0]['Percentage']

region_order = coal_summary['Region'].tolist()

# Add labels to coal segments only
region_summary['Coal_Label'] = np.where(
    region_summary['Source'] == 'Coal',
    region_summary['Percentage'].map('{:.1f}%'.format),
    ''
)

# ── Step 1: Create 100% Stacked Bar Chart ─────────────────────
fig = px.bar(
    region_summary,
    x='Percentage',
    y='Region',
    color='Source',
    orientation='h',
    text='Coal_Label',
    barmode='stack',
    category_orders={
        'Region': region_order,
        'Source': ['Coal', 'Oil', 'Natural Gas']
    },
    color_discrete_map={
        'Coal': '#0072B2',
        'Oil': '#BDBDBD',
        'Natural Gas': '#737373'
    },
    title=f'{highest_region} Has the Highest Coal Share ({highest_share:.1f}%)'
)

# ── Step 2: Customize Chart ───────────────────────────────────
fig.update_traces(
    textposition='inside',
    insidetextanchor='middle'
)

fig.update_layout(
    font=dict(family='Arial', size=12),
    xaxis=dict(
        title='Share of Fossil Energy (%)',
        range=[0, 100]
    ),
    yaxis=dict(
        title='Region',
        categoryorder='array',
        categoryarray=region_order[::-1]
    ),
    margin=dict(l=10, r=10, t=55, b=10)
)

fig.show()